In [3]:
# -------------------------------
# Deep Q-Learning (DQN) complet
# -------------------------------

# Importation des bibliothèques
import gymnasium as gym
import random
import numpy as np
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim

# 2 Hyperparamètres
ENV_NAME = "CartPole-v1"
GAMMA = 0.99       # facteur de discount
LR = 0.001         # learning rate
BATCH_SIZE = 64
MEMORY_SIZE = 10000
EPSILON_START = 1.0
EPSILON_END = 0.01
EPSILON_DECAY = 0.995
TARGET_UPDATE = 10
EPISODES = 500

#  3 Définition du Q-Network
class QNetwork(nn.Module):
    def __init__(self, state_size, action_size):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(state_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_size)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)  # Q-values pour toutes les actions

# 4 Replay Memory
class ReplayMemory:
    def __init__(self, capacity):
        self.memory = deque(maxlen=capacity)
    
    def push(self, transition):
        self.memory.append(transition)
    
    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)
    
    def __len__(self):
        return len(self.memory)

# 2. Initialisation de l'environnement et des réseaux
env = gym.make(ENV_NAME)
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

policy_net = QNetwork(state_size, action_size)   # Réseau principal
target_net = QNetwork(state_size, action_size)   # Réseau cible
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()  # on ne l’entraîne pas directement

optimizer = optim.Adam(policy_net.parameters(), lr=LR)
memory = ReplayMemory(MEMORY_SIZE)
epsilon = EPSILON_START

# 6. Fonction pour choisir l'action (ε-greedy)
def select_action(state, epsilon):
    if random.random() < epsilon:
        return random.randrange(action_size)  # exploration
    else:
        with torch.no_grad():
            state = torch.FloatTensor(state)
            return policy_net(state).argmax().item()  # exploitation

# 7 Fonction d'entraînement
def train():
    if len(memory) < BATCH_SIZE:
        return
    
    # Échantillonnage aléatoire
    transitions = memory.sample(BATCH_SIZE)
    batch_state, batch_action, batch_reward, batch_next_state, batch_done = zip(*transitions)
    
    # Conversion en tensors
    batch_state = torch.FloatTensor(batch_state)
    batch_action = torch.LongTensor(batch_action).unsqueeze(1)
    batch_reward = torch.FloatTensor(batch_reward)
    batch_next_state = torch.FloatTensor(batch_next_state)
    batch_done = torch.FloatTensor(batch_done)
    
    # Q-values actuelles
    current_q = policy_net(batch_state).gather(1, batch_action)
    
    # Q-values cibles via Target Network
    next_q = target_net(batch_next_state).max(1)[0].detach()
    expected_q = batch_reward + (1 - batch_done) * GAMMA * next_q
    
    # Calcul de la perte et descente de gradient
    loss = nn.MSELoss()(current_q.squeeze(), expected_q)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# 8. Boucle principale d'apprentissage
for episode in range(EPISODES):
    state = env.reset()[0]
    total_reward = 0
    done = False
    
    while not done:
        action = select_action(state, epsilon)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        # Stockage de la transition
        memory.push((state, action, reward, next_state, float(done)))
        
        state = next_state
        total_reward += reward
        
        # Mise à jour du réseau principal
        train()
    
    # Décroissance de l’exploration
    epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)
    
    # Mise à jour périodique du Target Network
    if episode % TARGET_UPDATE == 0:
        target_net.load_state_dict(policy_net.state_dict())
    
    print(f"Episode {episode}, Total reward: {total_reward}, Epsilon: {epsilon:.3f}")

env.close()

#  Résumé :
# - Replay Memory pour stabiliser l’apprentissage
# - Target Network pour calculer des Q-targets stables
# - ε-greedy pour explorer et exploiter
# - Boucle principale : interaction avec l'environnement + mise à jour du DQN

Episode 0, Total reward: 15.0, Epsilon: 0.995
Episode 1, Total reward: 16.0, Epsilon: 0.990
Episode 2, Total reward: 18.0, Epsilon: 0.985


C:\Users\User\AppData\Local\Temp\ipykernel_31172\162188279.py:86: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\cb\pytorch_1000000000000\work\torch\csrc\utils\tensor_new.cpp:281.)
  batch_state = torch.FloatTensor(batch_state)


Episode 3, Total reward: 21.0, Epsilon: 0.980
Episode 4, Total reward: 36.0, Epsilon: 0.975
Episode 5, Total reward: 28.0, Epsilon: 0.970
Episode 6, Total reward: 18.0, Epsilon: 0.966
Episode 7, Total reward: 38.0, Epsilon: 0.961
Episode 8, Total reward: 9.0, Epsilon: 0.956
Episode 9, Total reward: 49.0, Epsilon: 0.951
Episode 10, Total reward: 29.0, Epsilon: 0.946
Episode 11, Total reward: 18.0, Epsilon: 0.942
Episode 12, Total reward: 9.0, Epsilon: 0.937
Episode 13, Total reward: 26.0, Epsilon: 0.932
Episode 14, Total reward: 33.0, Epsilon: 0.928
Episode 15, Total reward: 20.0, Epsilon: 0.923
Episode 16, Total reward: 28.0, Epsilon: 0.918
Episode 17, Total reward: 24.0, Epsilon: 0.914
Episode 18, Total reward: 16.0, Epsilon: 0.909
Episode 19, Total reward: 16.0, Epsilon: 0.905
Episode 20, Total reward: 23.0, Epsilon: 0.900
Episode 21, Total reward: 18.0, Epsilon: 0.896
Episode 22, Total reward: 12.0, Epsilon: 0.891
Episode 23, Total reward: 24.0, Epsilon: 0.887
Episode 24, Total rewa

KeyboardInterrupt: 